In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

df = pd.read_csv("./DataSets/PlayGolfData.csv")
df.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25


### 1. Label Encoding

In [2]:
le = LabelEncoder()

df_le = df.copy()
df_le["Outlook"] = le.fit_transform(df_le["Outlook"])
df_le["Temperature"] = le.fit_transform(df_le["Temperature"])
df_le.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness
0,0,03-25,Mon,Mar,1,Dry,No,2,85
1,1,03-26,Tue,Mar,2,Humid,Yes,1,30
2,2,03-27,Wed,Mar,1,Dry,Yes,0,65
3,3,03-28,Thu,Mar,0,Dry,Yes,2,45
4,4,03-29,Fri,Mar,2,Humid,No,1,25


### 2. Ordinal Encoding

In [ ]:
ord_enc = OrdinalEncoder(categories=[
    ["Low", "High", "Extreme"],   # Temperature order
    ["Dry", "Humid"],            # Humidity
    ["No", "Yes"],               # Wind
    ["sunny", "overcast", "rainy"]  # Outlook (example order)
])

df_ord = df.copy()
df_ord[["Temperature","Humidity","Wind","Outlook"]] = ord_enc.fit_transform(
    df_ord[["Temperature","Humidity","Wind","Outlook"]]
)
df_ord.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness
0,0,03-25,Mon,Mar,1.0,0.0,0.0,0.0,85
1,1,03-26,Tue,Mar,0.0,1.0,1.0,2.0,30
2,2,03-27,Wed,Mar,1.0,0.0,1.0,1.0,65
3,3,03-28,Thu,Mar,2.0,0.0,1.0,0.0,45
4,4,03-29,Fri,Mar,0.0,1.0,0.0,2.0,25


### 3. One-Hot Encoding (Sklearn)

In [4]:
ohe = OneHotEncoder(sparse_output=False)

encoded = ohe.fit_transform(df[["Outlook"]])
df_ohe = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["Outlook"]))

df_ohe.head()

,Outlook_overcast,Outlook_rainy,Outlook_sunny
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0


### 4. One-Hot Encoding (Pandas Dummy)

In [5]:
df_dummy = pd.get_dummies(df, columns=["Outlook"])
df_dummy.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Crowdedness,Outlook_overcast,Outlook_rainy,Outlook_sunny
0,0,03-25,Mon,Mar,High,Dry,No,85,False,False,True
1,1,03-26,Tue,Mar,Low,Humid,Yes,30,False,True,False
2,2,03-27,Wed,Mar,High,Dry,Yes,65,True,False,False
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,45,False,False,True
4,4,03-29,Fri,Mar,Low,Humid,No,25,False,True,False


### 5. Custom Binary Encoding

In [6]:
mapping = {"sunny":1, "overcast":2, "rainy":3}

df_bin = df.copy()
df_bin["Outlook_bin"] = df_bin["Outlook"].map(mapping)

# Convert to binary (2-bit)
df_bin["Outlook_b1"] = df_bin["Outlook_bin"].apply(lambda x: format(x, '02b')[0])
df_bin["Outlook_b2"] = df_bin["Outlook_bin"].apply(lambda x: format(x, '02b')[1])

df_bin.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness,Outlook_bin,Outlook_b1,Outlook_b2
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85,1,0,1
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30,3,1,1
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65,2,1,0
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45,1,0,1
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25,3,1,1


### 6. Target Encoding

In [7]:
df_te = df.copy()

target_mean = df_te.groupby("Outlook")["Crowdedness"].mean()

df_te["Outlook_te"] = df_te["Outlook"].map(target_mean)
df_te.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness,Outlook_te
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85,75.000000
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30,33.750000
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65,68.333333
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45,75.000000
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25,33.750000


### 7. Custom Function Encoding

In [8]:
def temp_encoding(val):
    if val == "Low":
        return 1
    elif val == "High":
        return 2
    else:
        return 3

df_cf = df.copy()
df_cf["Temp_encoded"] = df_cf["Temperature"].apply(temp_encoding)
df_cf.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness,Temp_encoded
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85,2
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30,1
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65,2
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45,3
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25,1


### 8. Frequency Encoding

In [9]:
freq = df["Outlook"].value_counts()

df_freq = df.copy()
df_freq["Outlook_freq"] = df_freq["Outlook"].map(freq)
df_freq.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness,Outlook_freq
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85,5
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30,4
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65,3
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45,5
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25,4


### 9. Binary Encoding (Library-style idea)

In [10]:
df_be = df.copy()
df_be["Outlook_code"] = df_be["Outlook"].astype("category").cat.codes
df_be["Outlook_bin"] = df_be["Outlook_code"].apply(lambda x: format(x, '03b'))
df_be.head()

,Unnamed: 0,Date,Weekday,Month,Temperature,Humidity,Wind,Outlook,Crowdedness,Outlook_code,Outlook_bin
0,0,03-25,Mon,Mar,High,Dry,No,sunny,85,2,010
1,1,03-26,Tue,Mar,Low,Humid,Yes,rainy,30,1,001
2,2,03-27,Wed,Mar,High,Dry,Yes,overcast,65,0,000
3,3,03-28,Thu,Mar,Extreme,Dry,Yes,sunny,45,2,010
4,4,03-29,Fri,Mar,Low,Humid,No,rainy,25,1,001
